# 0.2 Environment setup

AI Agent Engineering · LogicStack — practice notebook

Run the cells in order. Change the code, break things, fix them. That is the practice.

Lesson page: https://logicstack.org/courses/ai-agent-engineering

In [ ]:
# Run this first: installs the exact versions this course uses.
!pip install -q "langchain==1.4.7" "langchain-core==1.6.13" "langchain-openai==1.7.2" "langchain-google-genai==4.4.2" "langgraph==1.2.16"

## Your API key - pick ONE option

Some cells in this notebook call a model, and you need your own API key. Run only ONE of the next two cells, whichever fits you:

**Option A - Gemini (free, recommended if you have no OpenAI credit):** Google AI Studio gives a free API key, no card needed. Open https://aistudio.google.com/apikey , create a key, then run the Gemini cell below. The Flash models run on a free tier with rate limits.

**Option B - OpenAI (paid):** create a key at https://platform.openai.com/api-keys and run the OpenAI cell below.

Whichever key you paste is stored only in this notebook session, never saved to the file.

In [ ]:
# Option A: Gemini (free). Run this cell OR the OpenAI one, not both.
import os, getpass
os.environ["GOOGLE_API_KEY"] = getpass.getpass("Paste your Gemini (AI Studio) API key (input stays hidden): ")

In [ ]:
# Option B: OpenAI (paid). Run this cell OR the Gemini one, not both.
import os, getpass
os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key (input stays hidden): ")

## Using Gemini instead of OpenAI in the cells below

If you ran the Gemini key cell, one more small step: wherever a cell below creates the model with `ChatOpenAI(...)`, swap that creation line for:

```python
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
```

(Use whatever variable name the cell used, for example `model = ...` instead of `llm = ...`.) Everything else stays the same: prompts, chains, parsers and tools do not care which provider the model came from. For embeddings, swap `OpenAIEmbeddings` for `GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")` from the same package. On the OpenAI path, ignore this note and run the cells exactly as written.

## Learning objectives

- Create an isolated Python environment for the course and install the pinned packages.
- Store API keys safely in a `.env` file that never gets committed.
- Set up the free local path (Ollama) so you can run lessons without paying for API calls.
- Verify the whole setup with one script before moving on.

## The idea in plain words

Every "the library is broken" thread you will ever read has the same shape: someone installed packages into their system Python, mixed two projects' dependencies, or pasted an API key into a chat. This lesson takes twenty minutes and prevents all of that. Do it properly once and you will not think about it again.

You have two paths for running models. Pick one, or set up both:

1. **Hosted API (OpenAI).** Easiest to start. Costs a little money per call, and your prompts leave your machine.
2. **Free hosted API (Gemini).** Google AI Studio gives you an API key at no cost and no card, and the Flash models run on a free tier with rate limits. Same idea as the OpenAI path, it just does not bill you while you learn. Step 3 shows how to get the key.
3. **Local models (Ollama).** Free after download, runs on your own machine, nothing leaves it. This is also the path that satisfies EU data-residency concerns. The trade-off: small local models are weaker than hosted ones, and they need a machine with enough RAM.

The course code is written so the model is one constructor argument. Swapping providers later is a two-line change, not a rewrite.

## Step 1: Python and a virtual environment

You need Python 3.11 or newer. The packages in this course support 3.10 through 3.14, but 3.11 is the floor.

In [ ]:
!python3 --version   # must print 3.11 or higher

Create a project folder and an isolated environment inside it:

In [ ]:
!mkdir -p ~/agents-course && cd ~/agents-course
!python3 -m venv .venv
!source .venv/bin/activate   # Windows: .venv\Scripts\activate

Your prompt should now show `(.venv)`. Every time you open a new terminal for this course, activate the environment first. Packages installed while it is active stay inside the project and cannot break your system Python.

(If you already use `uv`, the modern alternative is `uv venv && uv pip install ...`. Either tool is fine. The lessons show plain `pip` so nothing extra is required.)

## Step 2: Install the packages

With the environment active:

In [ ]:
!pip install "langchain==1.4.6" "langchain-openai==1.7.2" "langgraph==1.2.16" "python-dotenv"

Pinning matters here. LangChain and LangGraph ship new releases almost weekly, and examples written against one minor version can break against the next. The `==` pins make your setup match the lessons exactly. If you later upgrade, do it deliberately and re-run the verification script in Step 5.

For the local Ollama path, also install:

In [ ]:
!pip install "langchain-ollama"

## Step 3: API keys, the safe way

Never put an API key directly in your code. Never paste one into a chat, a forum, or a screenshot. Keys go in a `.env` file in the project folder, and that file never gets committed to git.

Create `~/agents-course/.env`:

In [ ]:
OPENAI_API_KEY=sk-your-key-here
GOOGLE_API_KEY=your-gemini-key-here

Load it in Python with `python-dotenv`:

In [ ]:
from dotenv import load_dotenv
load_dotenv()  # reads .env from the current folder into environment variables

Get your key from the OpenAI dashboard (platform.openai.com). Two money rules that will save you pain:

- **Set a spending limit** on your API account on day one. A runaway agent loop can burn through credit fast; a monthly cap turns that into a lesson instead of a bill.
- **Start on the cheapest chat model** that does the job. Lesson code uses small models deliberately.

**The free option: a Gemini key.** If you do not want to pay while learning, get a key from Google AI Studio (aistudio.google.com, then "Get API key"). It is free, needs no card, and the Flash models (for example `gemini-2.5-flash`) run on a free tier with per-day rate limits, which is plenty for lesson-sized practice. Put the key in the same `.env` file as `GOOGLE_API_KEY`. Two honest caveats: the free tier is rate-limited, so a heavy agent loop can hit the wall and has to wait for the quota to reset; and on the free tier Google may use prompts to improve its products, so do not send anything sensitive through it. Lesson 1.4 shows how the same lesson code runs on Gemini with a one-line model swap.

If you use git in this folder, create `.gitignore` now and add `.env` to it:

In [ ]:
.venv/
.env
__pycache__/

## Step 4: The local path with Ollama

Install Ollama from ollama.com (one download, or the install script on Linux/macOS). Then pull a small chat model and start the server:

In [ ]:
!ollama pull llama3.1:8b
!ollama serve   # usually already running in the background after install

`llama3.1:8b` needs roughly 8 GB of RAM free and runs fine on most modern laptops without a GPU. It is weaker than a hosted flagship model, but it is free, private, and good enough for everything in Modules 0 and 1. The LangChain wrapper is:

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.1:8b")
print(llm.invoke("Say hello in one sentence.").content)

Ollama serves an OpenAI-compatible API on `http://localhost:11434`, which means most tools that talk to OpenAI can be pointed at it. Your data never leaves your machine, which is the simplest possible answer to data-residency questions.

## Step 5: Verify everything

Save this as `check_setup.py` and run it with the environment active:

In [ ]:
import sys
from dotenv import load_dotenv

load_dotenv()

print("Python:", sys.version.split()[0])
assert sys.version_info >= (3, 11), "Need Python 3.11+"

import langchain, langchain_openai, langgraph
print("langchain:", langchain.__version__)
print("langchain-openai:", langchain_openai.__version__)
print("langgraph:", langgraph.__version__)

import os
if os.getenv("OPENAI_API_KEY"):
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model="gpt-4o-mini")
    print("OpenAI path:", llm.invoke("Reply with the word OK.").content.strip())
else:
    print("OPENAI_API_KEY not set, skipping hosted check (fine if you use Ollama).")

try:
    from langchain_ollama import ChatOllama
    local = ChatOllama(model="llama3.1:8b")
    print("Ollama path:", local.invoke("Reply with the word OK.").content.strip())
except Exception as e:
    print("Ollama check skipped:", e)

print("Setup looks good.")

At least one of the two model paths must print OK. If neither does, stop here and fix it before Lesson 0.3.

## What can go wrong

- **Wrong interpreter.** You installed the packages but Python still cannot find them. Cause: you ran `pip` in one terminal and `python` in another without activating `.venv` in both. Always check `which python` (or `where python` on Windows) points inside `.venv`.
- **Key not loaded.** `OPENAI_API_KEY` is set in your shell but the script cannot see it. Cause: you forgot `load_dotenv()`, or the `.env` file is in a different folder than where you run the script. `load_dotenv()` reads from the current working directory.
- **Ollama model missing.** `ChatOllama` fails with a not-found error. Cause: you skipped `ollama pull llama3.1:8b`. Pull first, then run.
- **Ollama is slow.** First response takes a minute. That is the model loading into RAM, not a bug. Later calls are faster. If it is unusably slow, your machine is short on RAM; use the hosted path instead.
- **Committed a key.** If `.env` ever ends up in git, the key is compromised even if you delete it later. Revoke it in the provider dashboard immediately and generate a new one. Prevention (`.gitignore`) beats cleanup.
- **Unpinned installs.** `pip install langchain` today and `pip install langchain` in three months can give different minor versions with different behaviour. Pin, as above.

## Check your understanding

**1. Why use a virtual environment instead of installing into system Python?**
It isolates the project's dependencies. Different projects can need different versions of the same package, and system-wide installs can break tools your OS depends on.

**2. Where does your API key live, and what two places must it never appear?**
In a `.env` file in the project folder, loaded with `python-dotenv`. It must never appear in source code or in git (use `.gitignore`), and never be pasted into chats, forums, or screenshots.

**3. What does Ollama give you that a hosted API does not?**
Free local inference with no per-call cost, and data that never leaves your machine, which is the simplest answer to EU data-residency requirements. The trade-off is weaker models and local RAM/CPU requirements.

**4. You run the verify script and get `ModuleNotFoundError: No module named 'langchain'`. What is the most likely cause?**
The virtual environment is not active in this terminal (or you installed into a different interpreter). Activate `.venv` and reinstall, or check `which python`.

**5. Why pin exact versions with `==` instead of installing the latest?**
LangChain and LangGraph release almost weekly and behaviour can change between minor versions. Pins make your environment match the lessons exactly, so examples behave as written.

## Done

Back to the lesson: https://logicstack.org/courses/ai-agent-engineering

When the code in this notebook feels easy, mark the lesson complete and move to the next one.